# Selection audit — what is knowable before any retrieval

The cell fill picks *which* queries the dataset contains. Whether those queries
produce clean route labels needs corpora, embeddings and retrieval — none of
which this notebook touches.

Everything below is a property of the **selection itself**, computable from
`cell_selection.parquet` and the feature catalog. Each section names the
question it answers and, where it exists, the number the old slice fill scored.

In [2]:
import sys
sys.path.insert(0, ".")

import pandas as pd

from composition.cells import CELLS
from composition.recipe import Recipe

CATALOG = "data/feature_table/catalog.parquet"
NEW = "data/composition/cell_selection.parquet"
OLD = "data/composition/selection.parquet"
REPORT = "data/composition/cell_report.parquet"

key = lambda f: f["dataset"] + "|" + f["query_id"].astype(str)
catalog = pd.read_parquet(CATALOG).assign(key=lambda d: key(d))
new = pd.read_parquet(NEW).assign(key=lambda d: key(d))
old = pd.read_parquet(OLD).assign(key=lambda d: key(d))
report = pd.read_parquet(REPORT)
recipe = Recipe()

print(f"catalog {len(catalog):,} rows / {catalog['dataset'].nunique()} lanes")
print(f"new selection {len(new):,} rows / {new['dataset'].nunique()} lanes / {new['cell'].nunique()} cells")
print(f"n_per_route = {recipe.n_per_route}  ->  decisive target {len(CELLS) * recipe.n_per_route * 2:,}")

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


catalog 440,534 rows / 42 lanes
new selection 57,653 rows / 42 lanes / 44 cells
n_per_route = 200  ->  decisive target 17,600


In [3]:
new['cell'].unique()

<ArrowStringArray>
[              'bare_concept_token',       'short_grammatical_question',
     'stopword_saturated_midlength',      'verbose_grammatical_request',
      'extreme_length_pasted_query',     'high_morphological_variation',
     'deep_nesting_single_sentence',     'multi_statement_context_dump',
            'wide_flat_enumeration',           'keyword_telegram_short',
        'negation_bearing_question',         'comparative_multi_entity',
  'conversational_courtesy_wrapper',                     'bare_acronym',
          'acronym_inside_question',             'pasted_code_fragment',
       'code_symbol_named_in_prose',           'symbol_pile_no_grammar',
            'math_notation_present',           'boolean_operator_query',
       'relative_temporal_no_dates',           'datetime_token_present',
         'version_pinned_technical',            'status_code_idf_split',
       'rare_key_buried_in_chatter',                'bare_number_token',
   'number_inside_natural_questi

## 1. Did the lane confound break?

The defect the redesign exists to fix: route outcome was a property of *which
corpus* a query came from. A cell drawn mostly from one corpus teaches that
corpus, not the archetype. Under the old fill, 12 of 23 populated cells drew
≥80% of their rows from a single lane.

One caveat this cannot answer: spread is necessary, not sufficient. Whether
decorrelation actually changes what the router learns needs labels.

In [4]:
def top_lane_share(frame, group, floor=20):
    out = {}
    for name, g in frame.groupby(group):
        if len(g) < floor:
            continue
        counts = g["dataset"].value_counts()
        out[name] = counts.iloc[0] / counts.sum()
    return pd.Series(out, name="top_lane_share")

new_share = top_lane_share(new, "cell")
old_share = top_lane_share(old, "slice")

summary = pd.DataFrame({
    "groups": [len(new_share), len(old_share)],
    "mean": [new_share.mean(), old_share.mean()],
    "median": [new_share.median(), old_share.median()],
    "worst": [new_share.max(), old_share.max()],
    ">=80% one lane": [(new_share >= 0.8).sum(), (old_share >= 0.8).sum()],
}, index=["new (per cell)", "old (per slice)"])
display(summary.round(2))

new_share.sort_values(ascending=False).head(10).to_frame().style.format("{:.0%}")

,groups,mean,median,worst,>=80% one lane
new (per cell),35,0.44,0.29,0.99,7
old (per slice),4,0.47,0.45,0.66,0


,top_lane_share
bare_number_token,99%
web_locator_token,97%
bare_machine_token,93%
registry_structured_identifier,93%
logistics_catalog_token,87%
geo_coordinate_postal,84%
bare_acronym,83%
rare_key_buried_in_chatter,80%
status_code_idf_split,77%
short_quantified_spec,71%


## 2. Which register did we end up with?

The old selection was criticised for being natural-language-heavy, so
auto-fusion had nothing to bite on and the canonical sparse archetypes were
thin. Wave 2 added WebFAQ, GooAQ and CLERC — all natural-question registers —
so this is where the acquisition could have pulled the wrong way.

In [5]:
identifier_columns = [c for c in catalog.columns if c.startswith("structured_identifiers.")]
catalog["identifier_spans"] = catalog[identifier_columns].sum(axis=1)

def register(keys, label):
    j = catalog[catalog["key"].isin(set(keys))]
    return {
        "selection": label,
        "rows_in_catalog": len(j),
        "median_words": j["length.length_words"].median(),
        "short_<=6w": (j["length.length_words"] <= 6).mean(),
        "nl_share": j["natural_language_signal.natural_language_share"].mean(),
        "any_identifier": (j["identifier_spans"] > 0).mean(),
        "lanes": j["dataset"].nunique(),
    }

pd.DataFrame([register(old["key"], "old d32"), register(new["key"], "new cells")]).set_index("selection").round(2)

,rows_in_catalog,median_words,short_<=6w,nl_share,any_identifier,lanes
selection,,,,,,
old d32,50000,7.0,0.47,0.32,0.44,21
new cells,51536,9.0,0.30,0.40,0.25,42


## 3. Which archetypes actually got supply?

`natural_rows` is what the cell could draw from; the draw is `2 * n_per_route`.
A cell below its draw is short, and one flagged `lane_share_infeasible` could
not reach the target lane share however it drew — its lanes are too thin.

In [7]:
draw = recipe.n_per_route * 2
view = report[[
    "cell", "lanes", "natural_rows", "top_share", "lane_share_cap",
    "lane_share_achieved", "lane_share_infeasible",
    "reused_dense", "reused_sparse", "labels_queued", "augmentation_rows",
]].copy()
view["short_of_draw"] = view["natural_rows"] < draw

print(f"draw per cell         {draw}")
print(f"cells short of draw   {view['short_of_draw'].sum()} of {len(view)}")
print(f"lane_share_infeasible {int(view['lane_share_infeasible'].sum())} of {len(view)}")
print(f"augmentation rows     {int(view['augmentation_rows'].sum())}")
view.sort_values("natural_rows").head(12)

draw per cell         400
cells short of draw   22 of 44
lane_share_infeasible 27 of 44
augmentation rows     7084


,cell,lanes,natural_rows,top_share,lane_share_cap,lane_share_achieved,lane_share_infeasible,reused_dense,reused_sparse,labels_queued,augmentation_rows,short_of_draw
33,legal_citation_canonical,0,0,0.000,1.0,0.0,True,0,0,0,400,True
21,datetime_token_present,1,1,1.000,1.0,0.0,True,0,0,1,399,True
35,bibliographic_catalog_identifier,1,2,1.000,1.0,0.0,True,0,0,2,398,True
34,bio_clinical_identifier,2,2,0.500,1.0,0.0,True,0,0,2,398,True
42,business_temporal_reference,1,2,1.000,1.0,0.0,True,0,0,2,398,True
17,symbol_pile_no_grammar,1,2,1.000,1.0,0.0,True,0,0,2,398,True
43,single_token_char_blob,2,2,0.500,1.0,0.0,True,0,0,2,398,True
36,travel_transport_code,2,3,0.667,1.0,0.0,True,0,0,3,397,True
40,standards_compliance_lookup,1,6,1.000,1.0,0.0,True,0,0,6,394,True
19,boolean_operator_query,6,7,0.286,1.0,0.0,True,0,0,6,393,True


## 4. What did each cell actually match?

The aggregate numbers cannot tell you whether a predicate caught what it meant
to. This prints real queries per cell alongside the measured band values that
made each row match, so a cell matching noise is visible rather than inferred.

Two were already found this way: `uri_in_query` matches long questions that
merely *mention* a URL (its predicate has no length bound), and
`opaque_token_any_domain` matched `"how to get 277 V AC ?"` because the
`http_status_code` bank fired on `277`.

In [13]:
by_name = {c.name: c for c in CELLS}
measured = new.merge(catalog.drop(columns=["dataset", "query_id"]), on="key", how="left")


def samples(cell_name, n=4, width=90):
    """Queries a cell claimed, with the band values that made them match."""
    cell = by_name[cell_name]
    bands = list(cell.predicate) + list(cell.any_of)
    rows = measured[measured["cell"] == cell_name].head(n)
    clip = lambda t: t[:width] + ("…" if len(t) > width else "")
    return pd.DataFrame([
        {
            "lane": row["dataset"],
            "stage": row["stage"],
            "query": clip(" ".join(str(row["query"]).split())),
            **{band.member: row.get(band.column) for band in bands},
        }
        for _, row in rows.iterrows()
    ])


samples("keyword_telegram_short", width=500)

,lane,stage,query,natural_language_share,length_words,number
0,beir-nfcorpus,reused,Apthous Ulcer Mystery Solved,0.0,4.0,0.0
1,crumb-set-operation-entity-retrieval,reused,Non-fantasy parody comedy books.,0.0,5.0,0.0
2,msmarco-passage-dev,reused,research tech salary,0.0,3.0,0.0
3,rarb-math,reused,Problem: Compute $e^{\pi i}.$,0.0,5.0,0.0


In [14]:
# thinnest cells first — these are the augmentation-critical ones
for name in report.nsmallest(6, "natural_rows")["cell"]:
    print(f"=== {name}  predicts {'+'.join(by_name[name].predicts)}")
    frame = samples(name, n=3)
    print(frame.to_string(index=False) if len(frame) else "  (no rows)")
    print()

=== legal_citation_canonical  predicts sparse_only
  (no rows)

=== datetime_token_present  predicts dense_only+sparse_only
      lane     stage                                                query  datetime  length_words
webfaq-eng candidate 2021-03-16 - Call History Display Failure (Resolved)       1.0           8.0

=== symbol_pile_no_grammar  predicts sparse_only
 lane     stage                       query  code_identifier  natural_language_share  length_words
orcas candidate       www.workforce adp.cpm              2.0                     0.0           4.0
orcas candidate www.electrolux repairs.comm              2.0                     0.0           4.0

=== bio_clinical_identifier  predicts dense_only+sparse_only
      lane     stage            query  length_words  medical_code  genomic_accession
     orcas candidate  rs485 interface           2.0           1.0                0.0
webfaq-eng candidate What is an 855R?           4.0           0.0                1.0

=== bibliograph

In [15]:
# and the largest, where a loose predicate would quietly dominate the dataset
for name in report.nlargest(4, "labels_queued")["cell"]:
    print(f"=== {name}")
    print(samples(name, n=3).to_string(index=False))
    print()

=== stopword_saturated_midlength
                                lane  stage                                                                                      query  length_words  stopword_ratio  number
                       beir-nfcorpus reused                                   How to Reduce Exposure to Alkylphenols Through Your Diet           9.0        0.555556     0.0
                      crumb-legal-qa reused Is it unlawful to evict a tenant because of their gender identity? In the state of Arizona          17.0        0.529412     0.0
crumb-set-operation-entity-retrieval reused                                     Vietnam set novels that are not about war and conflict          10.0        0.500000     0.0

=== high_morphological_variation
                                lane  stage                                                 query  word_variation_share  natural_language_share  length_words
                       beir-nfcorpus reused How Doctors Responded to Being Named a 

### Rationale vs reality

Each cell carries the mechanism its author claimed. Read it against the samples
above: a rationale that does not describe what actually matched is a predicate
bug, not a labelling question, and it is fixable before any retrieval runs.

In [16]:
pd.set_option("display.max_colwidth", 160)
pd.DataFrame([
    {"cell": c.name, "predicts": "+".join(c.predicts), "rationale": c.rationale}
    for c in CELLS
]).set_index("cell")

,predicts,rationale
cell,,
bare_concept_token,dense_only,"One or two words naming a single concept give the embedding a clean, well-trained point in space, while BM25 holds a single medium-IDF term with no way to b..."
short_grammatical_question,dense_only+sparse_only,A short question whose token budget is mostly closed-class leaves two or three content words to carry the need. The route flips on whether those words are t...
stopword_saturated_midlength,dense_only+sparse_only,"Extends the buried-anchor shape into full-sentence territory: BM25 discards roughly half the tokens and scores on the three-to-five surviving content words,..."
verbose_grammatical_request,dense_only+pure_rrf,"A long grammatical ask gives dense a full intent to compress, while BM25 receives many medium-IDF content terms whose union over-matches tangential document..."
extreme_length_pasted_query,dense_only+sparse_only,"Sixty-plus words is no longer a question but a pasted passage. A single-vector embedding must average many facets and may truncate, while BM25 turns the sam..."
high_morphological_variation,dense_only+sparse_only,"When a large share of tokens are inflected or derived forms, exact matching depends on the analyzer's stemmer happening to conflate the query form with the ..."
deep_nesting_single_sentence,dense_only,A single deeply embedded sentence puts the meaning in the composition — who did what to whom under which condition — which a bag-of-words model flattens awa...
multi_statement_context_dump,dense_only+pure_rrf,"Several clauses mean background plus an actual ask: dense compresses the whole and risks diluting the ask, sparse harvests rare terms from every clause incl..."
wide_flat_enumeration,sparse_only+pure_rrf,"A coordination chain of three or more items is a multi-facet query — the known worst case for a single vector, which averages the facets into a centroid nea..."


## 5. Is the queue supply-bound or quota-bound?

`labels_wanted` is what the quota needs; `labels_queued` is what the cells could
actually offer. A large gap means the ceiling is per-cell supply under the lane
cap — which `recommended_sample` on the big boxes controls, not `n_per_route`.

In [17]:
wanted, queued = int(report["labels_wanted"].sum()), int(report["labels_queued"].sum())
print(f"labels_wanted  {wanted:,}")
print(f"labels_queued  {queued:,}  ({queued / wanted:.0%} of wanted)")
print(f"reused free    {int((new['stage'] == 'reused').sum()):,}")
print()
print("box sample caps vs what exists upstream:")
pd.DataFrame([
    {"box": "clerc", "sampled": 50_000, "available": 327_414},
    {"box": "gooaq", "sampled": 50_000, "available": 3_031_709},
    {"box": "webfaq-eng", "sampled": 50_000, "available": 5_278_725},
    {"box": "orcas", "sampled": 100_000, "available": 10_405_342},
]).assign(share=lambda d: (d["sampled"] / d["available"]).map("{:.1%}".format)).set_index("box")

labels_wanted  406,776
labels_queued  46,250  (11% of wanted)
reused free    1,794

box sample caps vs what exists upstream:


,sampled,available,share
box,,,
clerc,50000,327414,15.3%
gooaq,50000,3031709,1.6%
webfaq-eng,50000,5278725,0.9%
orcas,100000,10405342,1.0%


## 6. Near-duplicate leakage

~5.5% of the old selection were cos>0.95 pairs, and the train/eval split was not
near-duplicate aware — so within-lane numbers may have been inflated by an
unknown amount. Measuring it now lets the first split be near-dup aware instead
of discovering it afterwards.

This is an **auxiliary** embedding of query strings via sentence-transformers —
unrelated to the fastembed retrieval stack, and it does not need any corpus.
Minutes, not hours. Set `RUN_NEAR_DUP = True` to execute.

In [18]:
RUN_NEAR_DUP = True

if RUN_NEAR_DUP:
    import numpy as np
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    texts = new["query"].fillna("").tolist()
    vectors = model.encode(texts, batch_size=256, show_progress_bar=True,
                           normalize_embeddings=True)
    # blocked cosine so 32K x 32K never materialises
    threshold, pairs = 0.95, 0
    for start in range(0, len(vectors), 2048):
        block = vectors[start:start + 2048] @ vectors.T
        block[np.arange(len(block)), np.arange(start, start + len(block))] = 0.0
        pairs += int((block > threshold).sum())
    print(f"cos>{threshold} pairs: {pairs // 2:,} over {len(vectors):,} queries")
else:
    print("skipped — set RUN_NEAR_DUP = True")

Batches: 100%|██████████| 226/226 [00:23<00:00,  9.57it/s]


cos>0.95 pairs: 12,026 over 57,653 queries
